### Install And Import

In [1]:
!pip install scikit-learn
!pip install tf_keras
!pip install pandas
!pip install numpy
!pip install seaborn
!pip install torch
!pip install sentencepiece
# !pip install tensorflow[and-cuda]




# Install required packages
!pip install -q transformers datasets
# Reinstall transformers and tensorflow to ensure proper setup for TF models
!pip uninstall -y transformers
!pip install -q transformers==4.41.0


Found existing installation: transformers 5.15.1
Uninstalling transformers-5.15.1:
  Successfully uninstalled transformers-5.15.1


In [2]:


import os



os.environ["TF_USE_LEGACY_KERAS"] = "1"

# Now import your packages
import tensorflow as tf
import tf_keras

I0000 00:00:1787562676.603009  144499 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787562676.639008  144499 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787562677.466709  144499 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:


# Check versions
# import tensorflow as tf
from transformers import __version__ as transformers_version

print(f"TensorFlow Version: {tf.__version__}")
print(f"Transformers Version: {transformers_version}")

/home/ashwin-ubuntu24/Desktop/tech/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TensorFlow Version: 2.21.0
Transformers Version: 4.41.0


In [4]:
### Download Data From Kaggle

#Connect Google drive to colab
# from google.colab import drive
# drive.mount('/gdrive')

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import transformers


### Data Processing

Load data

In [6]:
# df = pd.read_csv('labeledTrainData.tsv.zip', sep='\t')

df = pd.read_csv('train_(1)_(2)_(1).csv')

print(df.shape)

(44798, 4)


In [7]:
df.sample(n=5)

,ID,Review_Title,Review,Rating
6925,7582,Simply awesome,Nice,1
313,347,255\r\n,During unpacking When i see d box it was in ve...,0
7422,8135,Really Nice,Quantity okay bt lite thicknesses,1
11396,12516,Hated it!,Worst don't buy this camera,0
38139,41846,Value-for-money,Good,1


In [8]:
#Sentences and labels
sentences = df.Review.values
labels = df.Rating.values

## Tokenize data using Bert Tokenizer

In [9]:
# from transformers import AutoTokenizer

# Load tokenizer (AutoTokenizer is the modern way)
tokenizer = transformers.AutoTokenizer.from_pretrained('bert-base-uncased')



In [10]:
#tokenizer.vocab.items()

In [11]:
tokenized_texts = [tokenizer.tokenize(sent) for sent in sentences]

Token indices sequence length is longer than the specified maximum sequence length for this model (1197 > 512). Running this sequence through the model will result in indexing errors


In [12]:
sentences[0]

'fine at this price\r\n'

In [13]:
type(sentences[0])

str

In [14]:
len('good')

4

In [15]:
len(sentences[0].split(' '))

4

In [16]:
#Check tokenized text
print(tokenized_texts[0])

['fine', 'at', 'this', 'price']


In [17]:
len(tokenized_texts[0])

4

In [18]:
#We will use only first 200 tokens to do classification (this value can be changed)
max_length = 200
tokenized_texts = [sent[:max_length] for sent in tokenized_texts]

In [19]:
for i in range(len(tokenized_texts)):
    sent = tokenized_texts[i]
    sent = ['[CLS]'] + sent + ['[SEP]']
    tokenized_texts[i] = sent

In [20]:
print(tokenized_texts[0])

['[CLS]', 'fine', 'at', 'this', 'price', '[SEP]']


In [21]:
#Convert tokens into IDs
input_ids = [tokenizer.convert_tokens_to_ids(sent) for sent in tokenized_texts]

In [22]:
print(input_ids[0])

[101, 2986, 2012, 2023, 3976, 102]


In [23]:
#Pad our tokens which might be less than max_length size
input_ids = tf.keras.preprocessing.sequence.pad_sequences(input_ids,
                                                          maxlen=max_length+2,
                                                          truncating='post',
                                                          padding='post')

Split data between training and test

In [24]:
#80% data will be used for training while 20% will be used for test
trainX, testX, trainY, testY = train_test_split(input_ids, labels,
                                                test_size=0.2, random_state=12345)

Create Attention masks : Attention masks are useful to ignore padding tokens. Mask value will be set to 0 for padding tokens and 1 for actual tokens. We will create mask both for training and test data

In [25]:
# Create attention masks for training
train_attn_masks = []

# Create a mask of 1s for each token followed by 0s for padding
for seq in trainX:
  seq_mask = [float(i>0) for i in seq]
  train_attn_masks.append(seq_mask)

In [26]:
# Create attention masks for Test
test_attn_masks = []

# Create a mask of 1s for each token followed by 0s for padding
for seq in testX:
  seq_mask = [float(i>0) for i in seq]
  test_attn_masks.append(seq_mask)

In [27]:
print(train_attn_masks[100])

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

### Build Model

In [28]:
# from transformers import TFBertForSequenceClassification

model = transformers.TFBertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2,
    from_pt=True   # explicitly convert from PyTorch
)

I0000 00:00:1787562684.024023  144499 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6625 MB memory:  -> device: 0, name: NVIDIA T1000 8GB, pci bus id: 0000:01:00.0, compute capability: 7.5
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [29]:
model.summary()

Model: "tf_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  109482240 
                                                                 
 dropout_37 (Dropout)        multiple                  0         
                                                                 
 classifier (Dense)          multiple                  1538      
                                                                 
Total params: 109483778 (417.65 MB)
Trainable params: 109483778 (417.65 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [31]:
import tf_keras

# Prepare training: Compile tf.keras model with optimizer, loss and learning rate schedule
# Small constant to prevent division by zero
# Gradient clipping to prevent exploding gradients
optimizer = tf_keras.optimizers.Adam(learning_rate=3e-5, epsilon=1e-08, clipnorm=1.0)

In [32]:
# Define loss and metrics
loss = tf_keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf_keras.metrics.SparseCategoricalAccuracy('accuracy')]

In [33]:
# !pip install "tensorflow<2.16"

In [34]:
# 2. Compile the model
model.compile(optimizer=optimizer, loss=loss, metrics=metrics)

### Train Model

In [35]:
train_x_data = {'input_ids': np.array(trainX), 'attention_mask': np.array(train_attn_masks)}
test_x_data = {'input_ids': np.array(testX), 'attention_mask': np.array(test_attn_masks)}

In [36]:
model.fit(train_x_data, trainY,
         validation_data=(test_x_data, testY),
         batch_size=16,
         epochs=2)

Epoch 1/2


I0000 00:00:1787562760.720406  145548 service.cc:153] XLA service 0x7e01d0350880 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787562760.720429  145548 service.cc:161]   StreamExecutor [0]: NVIDIA T1000 8GB, Compute Capability 7.5 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1787562760.737733  145548 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1787562760.769526  145548 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1787562760.832571  145548 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


 152/2240 [=>............................] - ETA: 55:08 - loss: 0.1777 - accuracy: 0.9354

KeyboardInterrupt: 